# Objectif : predire le prix d'une course de Taxi

### Contexte du jeu de données Taxi Trips 2024 (Chicago)

**Description générale**
Le jeu de données Taxi Trips 2024 contient les courses de taxi enregistrées dans la ville de Chicago à partir de janvier 2024. Les informations proviennent du processus de reporting des chauffeurs et des compagnies de taxi auprès de la municipalité.

**Particularités et limites**
- Certaines informations sont modifiées afin de protéger la vie privée des passagers et chauffeurs.
- Les identifiants des taxis ne correspondent pas aux numéros réels de médaillon, mais restent cohérents pour un même véhicule.
- Les informations géographiques fines (comme les Census Tracts) sont absentes ou agrégées.
- Les horaires sont arrondis à des intervalles de 15 minutes.
- Toutes les courses ne sont pas nécessairement reportées, même si la ville estime que la majorité l’est.



In [68]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor

# Nettoyage Préliminaire de la Base

### 1. Compréhension initiale du dataset

In [69]:
df = pd.read_csv("../data/Taxi_Trips_(2024-)_20251120.csv",sep=",")

C:\Users\jbche\AppData\Local\Temp\ipykernel_23336\1583862138.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Taxi_Trips_(2024-)_20251120.csv",sep=",")


In [70]:
df.head(3)

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,4322c60fe5f724c188287f9f5f6d43b6111be75f,8e08799f421f71f00bb9ded6011b7e5369296773e81ad0...,11/01/2025 12:00:00 AM,11/01/2025 12:15:00 AM,"1,299",3.800,NaN,NaN,22.000,6.000,...,$0.00,$17.23,Mobile,Taxicab Insurance Agency Llc,41.923,-87.699,POINT (-87.6991553432 41.9227606205),41.944,-87.656,POINT (-87.6559981815 41.9442266014)
1,7b79b8e6d4bf639dc087758ec28e30e0d182a416,09fa5981c7e26333fda83e361dc043dfe136d3bd7940d2...,11/01/2025 12:00:00 AM,11/01/2025 12:15:00 AM,"1,036",4.910,NaN,NaN,6.000,8.000,...,$0.00,$17.95,Mobile,Choice Taxi Association Inc,41.944,-87.656,POINT (-87.6559981815 41.9442266014),41.900,-87.633,POINT (-87.6333080367 41.899602111)
2,95f4b1e917a6c040224d4cae05a76b972c58a5f3,b7f7dbb452c0fb980a0f2050a146147c1006fe5f34e3b0...,11/01/2025 12:00:00 AM,11/01/2025 12:15:00 AM,730,3.270,NaN,NaN,8.000,7.000,...,$0.00,$15.51,Mobile,5 Star Taxi,41.900,-87.633,POINT (-87.6333080367 41.899602111),41.923,-87.649,POINT (-87.6494887289 41.9226862843)


| Variable                       | Description synthétique |
|--------------------------------|--------------------------|
| Trip ID                        | Identifiant unique de la course. |
| Taxi ID                        | Identifiant anonymisé du taxi (cohérent, mais ne correspond pas au vrai medallion). |
| Trip Start Timestamp           | Date et heure de début de la course (arrondie au quart d’heure). |
| Trip End Timestamp             | Date et heure de fin de la course. |
| Trip Seconds                   | Durée totale de la course en secondes. |
| Trip Miles                     | Distance de la course en miles. |
| Pickup Census Tract            | Zone de recensement (census tract) où le passager est monté, si disponible. |
| Dropoff Census Tract           | Zone de recensement où le passager est descendu, si disponible. |
| Pickup Community Area          | Code de la communauté de Chicago où la prise en charge a eu lieu. |
| Dropoff Community Area         | Code de la communauté où la dépose a eu lieu. |
| Fare                           | Montant du tarif de la course (hors extras et pourboires). |
| Tips                           | Montant des pourboires. |
| Tolls                          | Montant des péages. |
| Extras                         | Frais supplémentaires (ex. surcharge). |
| Trip Total                     | Montant total payé par le passager. |
| Payment Type                   | Mode de paiement (cash, credit card, etc.). |
| Company                        | Nom de la compagnie de taxi ayant opéré la course. |
| Pickup Centroid Latitude       | Latitude du centroïde de la zone de prise en charge. |
| Pickup Centroid Longitude      | Longitude du centroïde de la zone de prise en charge. |
| Pickup Centroid Location       | Coordonnées combinées (point géographique) de la prise en charge. |
| Dropoff Centroid Latitude      | Latitude du centroïde de la zone de dépose. |
| Dropoff Centroid Longitude     | Longitude du centroïde de la zone de dépose. |
| Dropoff Centroid Location      | Coordonnées combinées (point géographique) de la dépose. |


In [71]:
print(f"-Le Nombre de colonnes est de : {df.shape[1]}\n-Le nombre de lignes est de {df.shape[0]}")

-Le Nombre de colonnes est de : 23
-Le nombre de lignes est de 12233748


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     object 
 1   Taxi ID                     object 
 2   Trip Start Timestamp        object 
 3   Trip End Timestamp          object 
 4   Trip Seconds                object 
 5   Trip Miles                  object 
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        object 
 11  Tips                        object 
 12  Tolls                       object 
 13  Extras                      object 
 14  Trip Total                  object 
 15  Payment Type                object 
 16  Company                     object 
 17  Pickup Centroid Latitude    float64
 18  Pickup Centroid Longitude   float64
 19  Pickup Centroid Loc

### 2. Analyse de la qualité des données

In [73]:
df.isna().sum().sort_values(ascending=False)

Dropoff Census Tract          6988028
Pickup Census Tract           6820182
Dropoff Community Area        1112901
Dropoff Centroid Longitude    1044112
Dropoff Centroid  Location    1044112
Dropoff Centroid Latitude     1044112
Pickup Community Area          340545
Pickup Centroid Location       334109
Pickup Centroid Latitude       334109
Pickup Centroid Longitude      334109
Trip Total                      28149
Fare                            28149
Extras                          28149
Tolls                           28149
Tips                            28149
Trip Seconds                     2313
Trip End Timestamp                132
Trip Miles                        101
Taxi ID                            11
Trip ID                             0
Trip Start Timestamp                0
Payment Type                        0
Company                             0
dtype: int64

In [74]:
num_price_cols = ["Trip Total", "Fare", "Tips", "Tolls", "Extras"]

for col in num_price_cols:
    print(f"Nettoyage de {col}...")
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)   # retirer les virgules thousands
        .str.replace("$", "", regex=False)   # retirer éventuels $
        .str.strip()                         # retirer espaces
        .replace("", pd.NA)                  # convertir vide -> NA
    )

    # Conversion finale en float
    df[col] = pd.to_numeric(df[col], errors="coerce")

Nettoyage de Trip Total...
Nettoyage de Fare...
Nettoyage de Tips...
Nettoyage de Tolls...
Nettoyage de Extras...


In [ ]:
df[num_price_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 5 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Trip Total  float64
 1   Fare        float64
 2   Tips        float64
 3   Tolls       float64
 4   Extras      float64
dtypes: float64(5)
memory usage: 466.7 MB


In [ ]:
df[num_price_cols].head(3)

,Trip Total,Fare,Tips,Tolls,Extras
0,17.230,13.000,3.730,0.000,0.000
1,17.950,15.290,2.160,0.000,0.000
2,15.510,15.010,0.000,0.000,0.000


In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
df[num_price_cols].describe()

,Trip Total,Fare,Tips,Tolls,Extras
count,12205599.000,12205599.000,12205599.000,12205599.000,12205599.000
mean,27.970,22.781,2.839,0.031,2.087
std,37.085,32.549,4.272,4.105,9.169
min,0.000,0.000,0.000,0.000,0.000
25%,10.250,8.500,0.000,0.000,0.000
50%,18.600,15.650,0.000,0.000,0.000
75%,42.310,34.250,4.000,0.000,2.000
max,9999.750,9999.750,400.000,5550.000,5559.500


- Les variables financières (Trip Total, Fare, Tips, Extras, Tolls) ainsi que les variables de distance (Trip Miles) et de durée (Trip Seconds) étaient initialement au format texte, à cause de symboles comme "$" ou ",".

- Ci-dessus nous convertissons chacune de ces colonnes en "float" afin de pouvoir effectuer des calculs et des statistiques.  

In [ ]:
num_trip_cols = ["Trip Miles", "Trip Seconds"]

for col in num_trip_cols:
    print(f"Nettoyage de {col}...")
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("", pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

Nettoyage de Trip Miles...
Nettoyage de Trip Seconds...


In [ ]:
df[num_trip_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 2 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Trip Miles    float64
 1   Trip Seconds  float64
dtypes: float64(2)
memory usage: 186.7 MB


In [ ]:
df[num_trip_cols].describe()

,Trip Miles,Trip Seconds
count,12233647.000,12231435.000
mean,6.700,1250.370
std,7.705,1630.032
min,0.000,0.000
25%,1.070,480.000
50%,3.220,926.000
75%,11.800,1696.000
max,3397.800,86400.000


In [ ]:
df[num_trip_cols].head()

,Trip Miles,Trip Seconds
0,3.800,1299.000
1,4.910,1036.000
2,3.270,730.000
3,0.980,334.000
4,12.900,1260.000


In [ ]:
geo_cols = ["Pickup Community Area", "Dropoff Community Area"]

for col in geo_cols:
    print(f"Nettoyage de {col}...")
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False) 
        .str.strip()
        .replace("", pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

Nettoyage de Pickup Community Area...
Nettoyage de Dropoff Community Area...


In [ ]:
geo_cols = ["Pickup Community Area", "Dropoff Community Area"]

for col in geo_cols:
    print(f"Nettoyage de {col}...")
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)  
        .str.strip()
        .replace("", pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

Nettoyage de Pickup Community Area...
Nettoyage de Dropoff Community Area...


###  Nettoyage des zones géographiques simplifiées (Community Areas)

Les colonnes Pickup Community Area et Dropoff Community Area représentent 
les zones administratives de Chicago (1 à 77) où commencent et se terminent les trajets.  
Ces deux variables sont essentielles pour capturer l’effet géographique sur le prix des courses, 
tout en évitant la complexité excessive des coordonnées GPS ou des Census Tracts.

Cependant, plusieurs problèmes étaient présents :
- des valeurs manquantes importantes (environ 340k pour pickup et 1.1M pour dropoff),
- des formats incohérents (valeurs sous forme de texte, parfois avec des virgules),
- des zones inconnues lorsque le taxi sort de Chicago ou lorsque la donnée est anonymisée.

Dans cette étape, je :
1. Nettoie le format (strip, suppression de caractères parasites) ;
2. Convertis les colonnes en type numérique float64 ;
3. Remplace les valeurs manquantes par -1, une nouvelle modalité indiquant 
   une zone géographique inconnue.

Cette approche permet de conserver tous les trajets tout en rendant les données cohérentes 
et exploitables pour l'analyse et la modélisation.


In [ ]:
df[geo_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 2 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Pickup Community Area   float64
 1   Dropoff Community Area  float64
dtypes: float64(2)
memory usage: 186.7 MB


In [ ]:
df[geo_cols].describe()

,Pickup Community Area,Dropoff Community Area
count,11893203.000,11120847.000
mean,35.545,26.097
std,26.144,20.598
min,1.000,1.000
25%,8.000,8.000
50%,32.000,28.000
75%,56.000,32.000
max,77.000,77.000


In [ ]:
df[geo_cols].head()

,Pickup Community Area,Dropoff Community Area
0,22.000,6.000
1,6.000,8.000
2,8.000,7.000
3,7.000,6.000
4,76.000,22.000


In [ ]:
date_cols = ["Trip Start Timestamp", "Trip End Timestamp"]

for col in date_cols:
    print(f"Conversion de {col} en datetime...")
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_cols].info()
df[date_cols].head()

Conversion de Trip Start Timestamp en datetime...


C:\Users\jbche\AppData\Local\Temp\ipykernel_23336\2773000789.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


Conversion de Trip End Timestamp en datetime...


C:\Users\jbche\AppData\Local\Temp\ipykernel_23336\2773000789.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 2 columns):
 #   Column                Dtype         
---  ------                -----         
 0   Trip Start Timestamp  datetime64[ns]
 1   Trip End Timestamp    datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 186.7 MB


,Trip Start Timestamp,Trip End Timestamp
0,2025-11-01,2025-11-01 00:15:00
1,2025-11-01,2025-11-01 00:15:00
2,2025-11-01,2025-11-01 00:15:00
3,2025-11-01,2025-11-01 00:00:00
4,2025-11-01,2025-11-01 00:15:00


**Conversion des colonnes temporelles en format datetime**

Les colonnes Trip Start Timestamp et Trip End Timestamp sont fournies dans le dataset
au format texte (string). Pour pouvoir analyser correctement la dimension temporelle des trajets 
et créer de futures variables (heure, jour de la semaine, période de la journée, etc.), 
je convertis ces deux colonnes en type datetime64.

Cette conversion permet :
- d’extraire facilement des informations temporelles (heure, jour, mois, année),
- d'étudier les heures de pointe et les comportements selon les périodes de la journée,
- de vérifier la cohérence entre la durée enregistrée (Trip Seconds) et la différence entre les timestamps.

 Les rares valeurs manquantes seront ensuite traitées lors du nettoyage.


In [ ]:
cat_cols = ["Payment Type", "Company"]

for col in cat_cols:
    print(f"Nettoyage de {col}...")
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()          # enlever espaces au début/fin
    )
    # On remplace les valeurs "nan" texte par NA réelle
    df[col] = df[col].replace(["nan", "NaN", "None", ""], pd.NA)
    df[col] = df[col].astype("category")

df[cat_cols].info()
for col in cat_cols:
    print(f"\nValeurs les plus fréquentes pour {col} :")
    print(df[col].value_counts(dropna=False).head(10))

Nettoyage de Payment Type...
Nettoyage de Company...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 2 columns):
 #   Column        Dtype   
---  ------        -----   
 0   Payment Type  category
 1   Company       category
dtypes: category(2)
memory usage: 23.3 MB

Valeurs les plus fréquentes pour Payment Type :
Payment Type
Credit Card    4638710
Cash           3233359
Mobile         2438598
Prcard         1419565
Unknown         462386
No Charge        34028
Dispute           7062
Prepaid             40
Name: count, dtype: int64

Valeurs les plus fréquentes pour Company :
Company
Flash Cab                            2553158
Taxi Affiliation Services            1666116
Taxicab Insurance Agency Llc         1417640
Sun Taxi                             1327798
City Service                         1196014
Chicago Independents                  746495
5 Star Taxi                           602775
Blue Ribbon Taxi Association          51

**Nettoyage des variables catégorielles (Payment Type, Company)**

Les colonnes catégorielles contenaient des espaces parasites, des chaînes 'nan' et d'autres incohérences.  
Nous nettouons ces valeurs (strip, remplacement des faux NA) puis nous convertissons les colonnes en category, ce qui :

- réduit la mémoire utilisée,
- prépare les données pour les futures étapes d’encodage,
- facilite l’analyse des distributions de modalités.

In [ ]:
for col in df.columns:
    print(f"'{col}'")

'Trip ID'
'Taxi ID'
'Trip Start Timestamp'
'Trip End Timestamp'
'Trip Seconds'
'Trip Miles'
'Pickup Census Tract'
'Dropoff Census Tract'
'Pickup Community Area'
'Dropoff Community Area'
'Fare'
'Tips'
'Tolls'
'Extras'
'Trip Total'
'Payment Type'
'Company'
'Pickup Centroid Latitude'
'Pickup Centroid Longitude'
'Pickup Centroid Location'
'Dropoff Centroid Latitude'
'Dropoff Centroid Longitude'
'Dropoff Centroid  Location'


In [ ]:
cols_to_drop = [
    "Pickup Census Tract",
    "Dropoff Census Tract",
    "Pickup Centroid Latitude",
    "Pickup Centroid Longitude",
    "Pickup Centroid Location",
    "Dropoff Centroid Latitude",
    "Dropoff Centroid Longitude",
    "Dropoff Centroid  Location"   # double espace ici par exemple
]

df = df.drop(columns=cols_to_drop,axis=1)


**Suppression des colonnes géographiques détaillées**

Le dataset contient plusieurs colonnes géographiques très précises 
(Census Tract, Centroid Latitude/Longitude, Centroid Location).  
Ces variables posent plusieurs problèmes :

- elles contiennent un très grand nombre de valeurs manquantes 
- elles sont souvent vides pour les trajets hors de Chicago 
- leur niveau de précision (tract, centroid) est trop élevé pour notre objectif 
- elles n'apportent pas d'information utile supplémentaire par rapport 
  aux variables plus simples comme Pickup Community Area et Dropoff Community Area.

Dans cette étape, je supprime donc ces colonnes afin de :
- réduire la taille du dataset,
- éliminer du bruit inutile,
- simplifier l'analyse exploratoire,
- et garder uniquement les informations géographiques réellement pertinentes.

Les Community Areas seront conservées, car elles décrivent les zones de prise en charge 
et de dépose de manière suffisamment informative et cohérente pour la modélisation.


In [ ]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12233748 entries, 0 to 12233747
Data columns (total 15 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Trip ID                 object        
 1   Taxi ID                 object        
 2   Trip Start Timestamp    datetime64[ns]
 3   Trip End Timestamp      datetime64[ns]
 4   Trip Seconds            float64       
 5   Trip Miles              float64       
 6   Pickup Community Area   float64       
 7   Dropoff Community Area  float64       
 8   Fare                    float64       
 9   Tips                    float64       
 10  Tolls                   float64       
 11  Extras                  float64       
 12  Trip Total              float64       
 13  Payment Type            category      
 14  Company                 category      
dtypes: category(2), datetime64[ns](2), float64(9), object(2)
memory usage: 1.2+ GB


(12233748, 15)

In [ ]:
df.isna().sum().sort_values(ascending=False)

Dropoff Community Area    1112901
Pickup Community Area      340545
Trip Total                  28149
Tolls                       28149
Tips                        28149
Fare                        28149
Extras                      28149
Trip Seconds                 2313
Trip End Timestamp            132
Trip Miles                    101
Taxi ID                        11
Trip ID                         0
Trip Start Timestamp            0
Payment Type                    0
Company                         0
dtype: int64

In [ ]:
df = df.dropna(subset=[
    "Trip Total",
    "Trip Miles",
    "Trip Seconds",
    "Fare",
    "Tips",
    "Tolls",
    "Extras"
])

In [ ]:
df["Pickup Community Area"] = df["Pickup Community Area"].fillna(-1)
df["Dropoff Community Area"] = df["Dropoff Community Area"].fillna(-1)

- Dans cette étape, nous traitons les valeurs manquantes des variables importantes.

- Pour les colonnes liées au prix (Trip Total, Fare, Tips, Tolls, Extras) ainsi que les colonnes de distance et de durée (Trip Miles, Trip Seconds), les valeurs manquantes représentent moins de 0,3 % du dataset. Comme ces variables sont essentielles pour la modélisation et qu'une imputation menera à des resultats moins fiables nous préférons de  supprimer ces lignes.

- Pour les colonnes géographiques simples (Pickup Community Area et Dropoff Community Area), les valeurs manquantes correspondent principalement à des trajets réalisés hors Chicago ou à des zones anonymisées. Plutôt que de supprimer plus d’un million de lignes, nous imputons ces valeurs manquantes avec une nouvelle catégorie –1, représentant une zone inconnue ("Unknown Area").

- Cette approche permet :
  - de conserver un maximum de données,
  - d’éviter de perdre des trajets valides,
  - de préserver une information exploitable pour la modélisation.


In [ ]:
df.isna().sum().sort_values(ascending=False)

Taxi ID                   11
Trip ID                    0
Trip Start Timestamp       0
Trip End Timestamp         0
Trip Seconds               0
Trip Miles                 0
Pickup Community Area      0
Dropoff Community Area     0
Fare                       0
Tips                       0
Tolls                      0
Extras                     0
Trip Total                 0
Payment Type               0
Company                    0
dtype: int64

**Regardons s'il existe des doublons :**

In [ ]:
df.duplicated().sum()

np.int64(0)

**On enregistre la premiere version de la base**

In [ ]:
df.to_csv("../data/df_taxi_v1.csv")

# Analyse Exploratoire (EDA)

In [ ]:
df = pd.read_csv("../data/df_taxi_v1.csv")

| Variable                       | Description synthétique |
|--------------------------------|--------------------------|
| Trip ID                        | Identifiant unique de la course. |
| Taxi ID                        | Identifiant anonymisé du taxi (cohérent, mais ne correspond pas au vrai medallion). |
| Trip Start Timestamp           | Date et heure de début de la course (arrondie au quart d’heure). |
| Trip End Timestamp             | Date et heure de fin de la course. |
| Trip Seconds                   | Durée totale de la course en secondes. |
| Trip Miles                     | Distance de la course en miles. |
| Pickup Census Tract            | Zone de recensement (census tract) où le passager est monté, si disponible. |
| Dropoff Census Tract           | Zone de recensement où le passager est descendu, si disponible. |
| Pickup Community Area          | Code de la communauté de Chicago où la prise en charge a eu lieu. |
| Dropoff Community Area         | Code de la communauté où la dépose a eu lieu. |
| Fare                           | Tarif de base de la course (ne comprend ni les pourboires, ni les péages, ni les extras). |
| Tips                           | Montant des pourboires laissés par le passager. |
| Tolls                          | Montant des péages facturés pendant la course. |
| Extras                         | Frais supplémentaires (ex. surcharge carburant, surcharge aéroport). |
| Trip Total                     | Montant total payé par le passager (Fare + Tips + Tolls + Extras). |
| Payment Type                   | Mode de paiement utilisé (espèces, carte bancaire, etc.). |
| Company                        | Nom de la compagnie de taxi ayant réalisé la course. |
| Pickup Centroid Latitude       | Latitude du centroïde de la zone de prise en charge. |
| Pickup Centroid Longitude      | Longitude du centroïde de la zone de prise en charge. |
| Pickup Centroid Location       | Coordonnées combinées (point géographique) de la prise en charge. |
| Dropoff Centroid Latitude      | Latitude du centroïde de la zone de dépose. |
| Dropoff Centroid Longitude     | Longitude du centroïde de la zone de dépose. |
| Dropoff Centroid Location      | Coordonnées combinées (point géographique) de la dépose. |


In [ ]:
df.info() # rtes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12204111 entries, 0 to 12204110
Data columns (total 16 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Unnamed: 0              int64  
 1   Trip ID                 object 
 2   Taxi ID                 object 
 3   Trip Start Timestamp    object 
 4   Trip End Timestamp      object 
 5   Trip Seconds            float64
 6   Trip Miles              float64
 7   Pickup Community Area   float64
 8   Dropoff Community Area  float64
 9   Fare                    float64
 10  Tips                    float64
 11  Tolls                   float64
 12  Extras                  float64
 13  Trip Total              float64
 14  Payment Type            object 
 15  Company                 object 
dtypes: float64(9), int64(1), object(6)
memory usage: 1.5+ GB
